In [1]:

import pandas as pd
import numpy as np

file_path = "fobhav_noisy.csv"

cols = [
    'INSTRUMENT','SYMBOL','EXPIRY_DT','STRIKE_PR','OPTION_TYP',
    'OPEN','HIGH','LOW','CLOSE','SETTLE_PR',
    'CONTRACTS','VAL_INLAKH','OPEN_INT','CHG_IN_OI','TIMESTAMP'
]

df = pd.read_csv(file_path, usecols=cols, low_memory=True)

print("Initial Shape:", df.shape)

df.dropna(how='all', inplace=True)

df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
df['EXPIRY_DT'] = pd.to_datetime(df['EXPIRY_DT'], errors='coerce')

df.dropna(subset=['TIMESTAMP','CLOSE'], inplace=True)

df = df.sort_values(by='TIMESTAMP')
df.reset_index(drop=True, inplace=True)

df.fillna(method='ffill', inplace=True)

float_cols = ['OPEN','HIGH','LOW','CLOSE','SETTLE_PR','STRIKE_PR','VAL_INLAKH']
int_cols = ['CONTRACTS','OPEN_INT','CHG_IN_OI']

for col in float_cols:
    df[col] = df[col].astype('float32')

for col in int_cols:
    df[col] = df[col].astype('int32')

df['SYMBOL'] = df['SYMBOL'].astype('category')
df['INSTRUMENT'] = df['INSTRUMENT'].astype('category')
df['OPTION_TYP'] = df['OPTION_TYP'].astype('category')

df = df[df['SYMBOL'] == 'NIFTY']

df['TARGET'] = (df['CLOSE'].shift(-1) > df['CLOSE']).astype('int8')
df.dropna(inplace=True)

df['PRICE_CHANGE'] = (df['CLOSE'] - df['OPEN']).astype('float32')
df['HIGH_LOW_DIFF'] = (df['HIGH'] - df['LOW']).astype('float32')
df['OI_CHANGE'] = df['CHG_IN_OI']
df['VOLUME'] = df['CONTRACTS']

df['HOUR'] = df['TIMESTAMP'].dt.hour.astype('int8')
df['DAY'] = df['TIMESTAMP'].dt.day.astype('int8')
df['MONTH'] = df['TIMESTAMP'].dt.month.astype('int8')

df['DAYS_TO_EXPIRY'] = (df['EXPIRY_DT'] - df['TIMESTAMP']).dt.days.astype('int16')

features = [
    'OPEN','HIGH','LOW','CLOSE',
    'OPEN_INT','OI_CHANGE','VOLUME',
    'PRICE_CHANGE','HIGH_LOW_DIFF',
    'HOUR','DAY','MONTH',
    'DAYS_TO_EXPIRY'
]

X = df[features]
y = df['TARGET']

print("\nFinal Shape:", X.shape)
print("Memory Usage (MB):", df.memory_usage().sum() / 1e6)

print("\nTarget Distribution:")
print(y.value_counts()) 

Initial Shape: (57600000, 15)


/tmp/ipykernel_55/3068676021.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
/tmp/ipykernel_55/3068676021.py:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)



Final Shape: (3570859, 13)
Memory Usage (MB): 321.463226

Target Distribution:
TARGET
0    1798062
1    1772797
Name: count, dtype: int64


In [2]:

df['MA_5'] = df['CLOSE'].rolling(window=5).mean()
df['MA_10'] = df['CLOSE'].rolling(window=10).mean()

df['MOMENTUM'] = df['CLOSE'] - df['CLOSE'].shift(5)
xd
df['VOLATILITY'] = df['HIGH'] - df['LOW']

df['OI_RATIO'] = df['CHG_IN_OI'] / (df['OPEN_INT'] + 1)

df['VOL_SPIKE'] = df['CONTRACTS'] / (df['CONTRACTS'].rolling(10).mean() + 1)

df['TREND'] = (df['MA_5'] > df['MA_10']).astype(int)

df.dropna(inplace=True)

features = [
    'OPEN','HIGH','LOW','CLOSE',
    'OPEN_INT','OI_CHANGE','VOLUME',
    'PRICE_CHANGE','HIGH_LOW_DIFF',
    'HOUR','DAY','MONTH',
    'DAYS_TO_EXPIRY',
    
    'MA_5','MA_10','MOMENTUM',
    'VOLATILITY','OI_RATIO','VOL_SPIKE','TREND'
]

X = df[features]
y = df['TARGET']

print("New Shape:", X.shape)

New Shape: (3570850, 20)


In [3]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False 
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred)

Train Shape: (2856680, 20)
Test Shape: (714170, 20)

Accuracy: 0.650646204685159

Classification Report:

              precision    recall  f1-score   support

           0       0.63      0.73      0.68    361010
           1       0.67      0.57      0.62    353160

    accuracy                           0.65    714170
   macro avg       0.65      0.65      0.65    714170
weighted avg       0.65      0.65      0.65    714170


Confusion Matrix:

[[263051  97959]
 [151539 201621]]
